In [13]:
from pymatgen.core import Structure, Element, Composition
from pymatgen.analysis.phase_diagram import PhaseDiagram, PDEntry,PDPlotter
from pymatgen.ext.matproj import MPRester
from pymatgen.io.vasp import Vasprun
import os
from tqdm.notebook import tqdm
import pandas as pd

from pymatgen.io.vasp.outputs import Vasprun, Outcar, Potcar
from pymatgen.entries.computed_entries import ComputedEntry, ComputedStructureEntry
from pymatgen.entries.compatibility import MaterialsProject2020Compatibility

In [3]:
par_el='run_full'
df = pd.read_pickle(f'pkl_files/df_{par_el}_pred_m3gnet.pkl')


In [4]:
import re
def sort_elements(formula):    # Extract elements using a regular expression
    elements = re.findall(r'[A-Z][a-z]*', formula)
    # Sort the elements alphabetically
    sorted_elements = ''.join(sorted(elements))
    return sorted_elements

# Apply the function to create a new column with sorted elements
df['Sorted_Elements'] = df['formula'].apply(sort_elements)

# Sort the dataframe by the sorted elements
df.sort_values(by='Sorted_Elements',inplace=True)

In [15]:
import numpy as np

In [40]:
# inds = []
# eahs = []
# cur_elements = {}

with MPRester('7JGUQgNZyOTTp8Tc') as mpr:
    for ind, entry in tqdm(df.iloc[6862:].iterrows(),total=len(df)-len(inds)):
        # try:

        try:
            path = f'/blue/hennig/jasongibson/diff_model/materials/{par_el}/mp_relaxed/{ind}'
            struct = Structure.from_file(f'{path}/CONTCAR')
            species =list(set([spec.name for spec in struct.species]))            
            vr = Vasprun(f'{path}/vasprun.xml') 
            elements = struct.symbol_set
        except:
            continue            
        if vr.converged:
            if cur_elements != elements:
                try:
                    entries = mpr.get_entries_in_chemsys(elements)
                except:
                    entries = mpr.get_entries_in_chemsys(elements)
                cur_elements = elements
                phasediagram = PhaseDiagram(entries)    
            else:
                print('same?')
            
            # potcar = Potcar.from_file(f'{entry.path}/POTCAR')
            # potcar_symbols = [f'PBE {p.symbol}' for p in potcar]
            pde = ComputedStructureEntry(vr.final_structure,vr.final_energy)
            # pde.parameters = {'run_type': 'GGA', 'potcar_symbols': potcar_symbols}  
            # pde = PDEntry(vr.final_structure.composition,vr.final_energy)
            # eah2 = phasediagram.get_e_above_hull(pde,allow_negative=True)
            # compat.process_entry(pde)
            eah = phasediagram.get_e_above_hull(pde,allow_negative=True)
            inds.append(int(ind))
            eahs.append(eah)
            if eah< 0:
                print(ind)
                print(eah)
            # print(f'eah_org: {eah2:.3f}\t eah_cor: {eah:.3f}')
        else:
            inds.append(int(ind))
            eahs.append(np.nan)
            print('BAD')

/home/jasongibson/miniconda3/envs/e3nn/lib/python3.9/site-packages/pymatgen/ext/matproj_legacy.py:167: UserWarning: You are using the legacy MPRester. This version of the MPRester will no longer be updated. To access the latest data with the new MPRester, obtain a new API key from https://materialsproject.org/api and consult the docs at https://docs.materialsproject.org/ for more information.
  warnings.warn(


  0%|          | 0/3127 [00:00<?, ?it/s]

/home/jasongibson/miniconda3/envs/e3nn/lib/python3.9/site-packages/pymatgen/io/vasp/outputs.py:327: UnconvergedVASPWarning: /blue/hennig/jasongibson/diff_model/materials/run_full/mp_relaxed/3908/vasprun.xml is an unconverged VASP run.
Electronic convergence reached: False.
Ionic convergence reached: True.
  warnings.warn(msg, UnconvergedVASPWarning)


BAD
same?
same?
same?
same?
same?
same?
same?
same?
same?
same?
same?
same?
same?
same?
same?
same?
same?
same?
same?
same?
same?
same?
same?
same?
same?
same?
same?
same?
same?
same?
same?
same?
same?
same?
same?
same?
same?
same?
same?
same?
same?
same?
same?
same?
same?
same?
same?
same?
same?
same?
same?
same?
same?
same?
same?
same?
same?
same?
same?
same?
same?
same?
same?
same?
same?
same?
same?
same?
same?
same?
same?
same?
same?
same?
same?
16172
-0.0023286908333339795
same?
same?
111509
-0.00024542166666563503
same?
same?
same?
same?
same?
same?
same?
same?
110298
-0.0021608200000002853
same?
same?
same?
same?
same?
same?
same?
same?
same?
same?
same?
same?
same?
same?
same?
same?
same?
101237
-0.009948737499998472
same?
same?
same?
same?
same?
same?
same?
same?
same?
168501
-0.0017161866666661751
same?
same?
same?
same?
same?
same?
same?
same?
same?
same?
same?
same?
same?
same?
same?
same?
4309
-0.013393281666663981
same?
same?
same?
same?
same?
same?
same?
same?
same?
same

In [51]:
df_eah = df.loc[inds]
df_eah['mp_eah'] = eahs

In [55]:
df_eah.to_pickle(f'pkl_files/df_{par_el}_pred_m3gnet_all_eah.pkl')
df_eah = df_eah.loc[df_eah.mp_eah<=0.2]#.head(25)#.sort_values('eah')
df_eah.to_pickle(f'pkl_files/df_{par_el}_pred_m3gnet_eah.pkl')